In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [7]:
loader = PyPDFLoader("../data/Application.pdf")
docs = loader.load()


In [8]:
len(docs)

4

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunk_data = splitter.split_documents(docs)

In [10]:
len(chunk_data)

8

In [11]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [12]:
vector_store = InMemoryVectorStore.from_documents(
    embedding=embeddings,
    documents=chunk_data
)

In [13]:
response = vector_store.similarity_search("name of the candidate")

In [14]:
print(response)
len(response)

[Document(id='8299265b-b029-4f06-9b98-ee8dd07e9993', metadata={'producer': 'mPDF 8.1.1', 'creator': 'PyPDF', 'creationdate': '2026-09-17T23:56:44+05:30', 'moddate': '2026-09-17T23:56:44+05:30', 'source': '../data/Application.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='17-09-2026 11:56:PM 2\nAADHAR DETAILS (\ue225थायी पता)\nCandidate Full\nName\nVignesh Vijay\nKumbhar\nAadhar Number XXXXXXXX6102 Date of Birth 2003-02-15\nRESERVATION\nDoes the candidate want to Apply for\nNRI? :\nउमेदवाराला NRI साठी अज\ue77a करायचा आहे \nका\ue98b:\nNO Is the candidate OCI card holder?उमेदवार OCI काड\ue77aधारक आहे\ue98b NO\nNationality of the\nCandidate :\nउमेदवाराचे रा\ue8d7ट\ue779ीय\ue8c9व :\nINDIAN\nApplied domicile of\nmaharasthra?:\nमहारा\ue8d7ट\ue779 अिधवास\n\ue955माणप\ue954ासाठी अज\ue77a \nक े ला आहे.:\n--\nConsider domicile\nof maharasthra?:\nमहारा\ue8d7ट\ue779ाचे अिधवास\n(डोिमसाईल) िवचारात\n\ue8c3यावे का\ue98b:\nYES\nApplied Category Of the\nCandidate:\nउमेदवाराची \ue957े

4

In [15]:
print(response[0].page_content)

17-09-2026 11:56:PM 2
AADHAR DETAILS (थायी पता)
Candidate Full
Name
Vignesh Vijay
Kumbhar
Aadhar Number XXXXXXXX6102 Date of Birth 2003-02-15
RESERVATION
Does the candidate want to Apply for
NRI? :
उमेदवाराला NRI साठी अज करायचा आहे 
का:
NO Is the candidate OCI card holder?उमेदवार OCI काडधारक आहे NO
Nationality of the
Candidate :
उमेदवाराचे राटीयव :
INDIAN
Applied domicile of
maharasthra?:
महाराट अिधवास
माणपासाठी अज 
क े ला आहे.:
--
Consider domicile
of maharasthra?:
महाराटाचे अिधवास
(डोिमसाईल) िवचारात
यावे का:
YES
Applied Category Of the
Candidate:
उमेदवाराची ेणी :
--
Consider Category Of the
Candidate:
उमेदवाराचा वग िवचारात या.:
OBC
Your Annual family Income:
तुमचे वािषक कौट ुंिबक उपन : 7,00,000 TO 8,00,000 Region of Residence:िनवासाचे े: 2-MAHANAGARPALIKA
Do you possess
Caste Certiﬁcate?:
तुमयाकडे जातीचा
दाखला आहे का:
AVAILABLE
Caste Certiﬁcate
Number:
जात माणप मांक:
MSC/SR/OBC/G/9424/2008
Caste Certiﬁcate
Inssuing District:
जात माणप जारी


### Agent = llm, tool, prompt


In [16]:
@tool
def retriver_tool(query:str):
    """
    This tool helps you to retrive the relevant data from the PDF documents. The PDF documents have data regarding college information
    """
    print("tool called", query) # to check how many times Agent called the tool, Agent should call the tool multiple times by breaking the user question. other wise there would be no sense for Agentic rag system
    relevant_chunks = vector_store.similarity_search(query=query, k = 4)
    context = ""
    for chunk in relevant_chunks:
        context += chunk.page_content + "\n\n"
        
    return context

In [17]:
res = retriver_tool("tell me about usha mittal college")  #this wont work as when we add tool decorator instead calling function we will have to invoke the tool
res

TypeError: 'StructuredTool' object is not callable

In [18]:
res = retriver_tool.invoke("HSC percentage of the candidate")
print(res)

17-09-2026 11:56:PM 3
Taluka of SSC/10th Std.Passing:
SSC/10वी इया उीण तालुका : KURLA
HSC QUALIFICATION
HSC/12th Equivalent Examination:
HSC/12वी समतुय परीा : PASS
HSC/12th or Equivalent
Examination Passing Year:
HSC/12वी िक ं वा समतुय परीा उीण 
वष:
2021
HSC/12th or Equivalent
Examination Roll No./Seat No.:
HSC/12वी िक ं वा समतुय परीा रोल
मांक/आसन मांक.:
M076027 State of HSC/12th Std.Passing:HSC/12वी इया उीण राय : MAHARASHTRA
District of HSC/12th Std.Passing:
HSC/12वी इया उीण िजहा : MUMBAI SUBURBAN Taluka of HSC/12th Std.Passing:HSC/12वी इया उीण तालुका : KURLA
SUBJECT MARKS OBTAINED MARKS OUT OF
PHYSICS 83 100
CHEMISTRY 84 100
BIOLOGY 88 100
ENGLISH 80 100
PCB TOTAL 255 300
PCBE Total 335 400
HSC Total 493 600
PCB PERCENTAGE 85 100
PCBE PERCENTAGE 83.75 100
HSC PERCENTAGE 82.17
ELIGIBILITY EXCEPTION
Select Speciﬁed Reservation
िनिदट आरण िनवडा NONE
APPLICATION FEES
I want to apply for
:
मला अज करायचा
आहे :
ONLY STATE
QUOTA
Select Courses :
अयासम िनवडा :

1

In [19]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [20]:
System_Prompt = """
        you are a helpful assistant that answers the user questions based on retrived context.
        Always use the `retriver_tool` tool for questions requiring extrernal knowledge
"""

### Main Agent

In [21]:
Agent = create_agent(
    model=llm,
    tools=[retriver_tool],
    system_prompt=System_Prompt
)

In [22]:
query = input("AI :")
Agent_response = Agent.invoke({'messages' : [{'role' : 'user', 'content' : query}]})
result = Agent_response['messages'][-1].content
print(result)


**Candidate’s name:** Vignesh Vijay Kumbhar  
**Date of birth:** 15 February 2003 (2003‑02‑15)
